In [17]:
import numpy as np

# Define dimensions
num_channels = 4  # Channels: 1, 2, 3, 4 (indices 0-3)
value_range = (0, 10)  # Values: 0 to 10 (11 values, indices 0-10)
z_range = (0, 48)  # z: 0 to 35 (36 values, indices 0-35)
y_range = (0, 343)  # y: 0 to 343 (344 values, indices 0-343)
x_range = (0, 680)  # x: 0 to 680 (681 values, indices 0-680)

# Calculate array dimensions
num_values = value_range[1] - value_range[0] + 1  # 11 values (0-10)
z_dim = z_range[1] - z_range[0] + 1  # 36 values (0-35)
y_dim = y_range[1] - y_range[0] + 1  # 344 values (0-343)
x_dim = x_range[1] - x_range[0] + 1  # 681 values (0-680)

# Calculate scaling factors
x_scale = x_dim / 172  # Old x_dim was 172
y_scale = y_dim / 87   # Old y_dim was 87

# Create 5D array initialized with zeros
# Shape: (channel, value, z, y, x)
data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

# Generate random values for each (z, y, x) position
for i in range(0, 3):
    for z in range(z_dim):
        # Channel 0: x ranges scaled from original 38+20*i to 43+20*i
        for y in range(y_dim):
            x_start = max(0, int((38+20*i) * x_scale))
            x_end = min(x_dim, int((43+20*i) * x_scale))
            for x in range(x_start, x_end):
                random_value = np.random.randint(6, 11)
                data[0, random_value, z, y, x] = 1
        # Channel 1: y ranges scaled from original 18+20*i to 23+20*i
        y_start = max(0, int((18+20*i) * y_scale))
        y_end = min(y_dim, int((23+20*i) * y_scale))
        for y in range(y_start, y_end):
            for x in range(x_dim):
                random_value = np.random.randint(6, 11)
                data[1, random_value, z, y, x] = 1
        # Channel 2: diagonal lines with slope 1 (y = x + c)
        strip_width = int(3 * max(x_scale, y_scale))  # Scaled strip width
        c_values = [int(-60 * x_scale), 0]  # Scaled c values
        for c in c_values:
            for y in range(y_dim):
                for x in range(x_dim):
                    if abs(y - x - c) <= strip_width:
                        random_value = np.random.randint(6, 11)
                        data[2, random_value, z, y, x] = 1
        # Channel 3: diagonal lines with slope -1 (y = -x + d)
        strip_width = int(3 * max(x_scale, y_scale))  # Scaled strip width
        d_values = [int(60 * y_scale), int(120 * y_scale)]  # Scaled d values
        for d in d_values:
            for y in range(y_dim):
                for x in range(x_dim):
                    if abs(y + x - d) <= strip_width:
                        random_value = np.random.randint(6, 11)
                        data[3, random_value, z, y, x] = 1

print(f"Created 5D data array with shape: {data.shape}")
print(f"Dimensions breakdown:")
print(f"  - Channels: {num_channels} (1, 2, 3, 4)")
print(f"  - Values: {num_values} (0 to {value_range[1]})")
print(f"  - Z: {z_dim} (0 to {z_range[1]})")
print(f"  - Y: {y_dim} (0 to {y_range[1]})")
print(f"  - X: {x_dim} (0 to {x_range[1]})")
print(f"\nData type: {data.dtype}")
print(f"Total elements: {data.size:,}")
print(f"Memory size: {data.nbytes / 1024 / 1024:.2f} MB")

# Save the data
np.save('groundtruth.npy', data)
print(f"\n✓ 5D data saved to: groundtruth.npy")


Created 5D data array with shape: (4, 11, 49, 344, 681)
Dimensions breakdown:
  - Channels: 4 (1, 2, 3, 4)
  - Values: 11 (0 to 10)
  - Z: 49 (0 to 48)
  - Y: 344 (0 to 343)
  - X: 681 (0 to 680)

Data type: int8
Total elements: 505,073,184
Memory size: 481.68 MB

✓ 5D data saved to: groundtruth.npy


In [18]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("groundtruth.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0
    "#1f78b4",    # channel 1
    "#b2df8a",     # channel 2
    "#33a02c",  # channel 3
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

for ch in range(n_channels):
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.1
max_points = 5000

for ch in range(n_channels):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,
            color=channel_colors[ch],
            opacity=0.8  # must be single number!
        ),
        name=f"Biomarker {ch}"
    ))

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Interactive View of Channels (Non-Isotropic: Z << X, Y)",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
    ),
    autosize=True,
    showlegend=True,
)

print(f"3D Visualization with real ranges (Non-Isotropic):")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


3D Visualization with real ranges (Non-Isotropic):
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y


In [19]:
import numpy as np

# ============================================================================
# STEP 1: Load the groundtruth data
# ============================================================================
data = np.load("groundtruth.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded data shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3)")
print(f"  Values: {n_values} (0-10)")
print(f"  Spatial dimensions: Z={n_z}, Y={y_dim}, X={x_dim}")

# ============================================================================
# STEP 2: Find voxel-level intersections (like sinusoid)
# Intersection = voxel where 2+ channels overlap
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 2: Finding voxel-level intersections")
print(f"{'='*60}")

# For each channel, sum over values axis to get presence map
# Input shape: (n_channels, n_values, n_z, y_dim, x_dim)
# Output shape: (n_channels, n_z, y_dim, x_dim)
channel_presence = np.sum(data, axis=1)

# Binary presence: 1 if channel has any data at that voxel
channel_binary = (channel_presence > 0).astype(np.int8)

# Count how many channels at each voxel; shape (n_z, y_dim, x_dim)
channels_per_voxel = np.sum(channel_binary, axis=0)

# Intersection voxels: 2 or more channels present
intersection_mask = channels_per_voxel >= 2

print(f"Channel presence shape: {channel_binary.shape}")
for ch in range(n_channels):
    print(f"  Channel {ch} non-zero voxels: {np.sum(channel_binary[ch] > 0):,}")
print(f"\nIntersection statistics (voxels with 2+ channels):")
print(f"  Voxels with 2 channels: {np.sum(channels_per_voxel == 2):,}")
print(f"  Voxels with 3 channels: {np.sum(channels_per_voxel == 3):,}")
print(f"  Voxels with 4 channels: {np.sum(channels_per_voxel == 4):,}")
print(f"  Total intersection voxels: {np.sum(intersection_mask):,}")

# ============================================================================
# STEP 3: Extract intersection voxel positions (like sinusoid)
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 3: Extracting intersection voxel positions")
print(f"{'='*60}")

intersection_coords = np.where(intersection_mask)
z_coords = np.array(intersection_coords[0], dtype=np.int64)
y_coords = np.array(intersection_coords[1], dtype=np.int64)
x_coords = np.array(intersection_coords[2], dtype=np.int64)

print(f"Total intersection voxels (GT): {len(z_coords):,}")
print(f"  These are voxels where 2 or more channels overlap")

# ============================================================================
# STEP 4: Ground truth summary & optional restrict + randomize (like sinusoid + limit)
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 4: Ground truth summary")
print(f"{'='*60}")

total_voxels = n_z * y_dim * x_dim
n_total = len(z_coords)
intersection_pct = (n_total / total_voxels) * 100

print(f"  Total voxels in volume: {total_voxels:,}")
print(f"  Intersection voxels: {n_total:,}")
print(f"  Intersection percentage: {intersection_pct:.2f}%")

# Optional: restrict to num_spots by random sampling (set to None to use ALL like sinusoid)
num_spots = 100       # None = use all intersections; int = randomly sample this many
random_seed = 42

if num_spots is not None and n_total > num_spots:
    np.random.seed(random_seed)
    idx = np.random.choice(n_total, size=num_spots, replace=False)
    z_coords = z_coords[idx]
    y_coords = y_coords[idx]
    x_coords = x_coords[idx]
    print(f"  Randomly sampled {num_spots} intersection voxels (seed={random_seed})")
else:
    print(f"  Using all {len(z_coords):,} intersections as GT")

# ============================================================================
# STEP 5: Create new array with 5 channels (like sinusoid)
# Channels 0-3: Copy from groundtruth.npy
# Channel 4: Mark intersection voxels as GT
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 5: Creating groundtruth_linear.npy with 5 channels")
print(f"{'='*60}")

ground_truth_linear = np.zeros((n_channels + 1, n_values, n_z, y_dim, x_dim), dtype=data.dtype)
ground_truth_linear[:n_channels] = data

print(f"Created new array with shape: {ground_truth_linear.shape}")
print(f"  Channels 0-3: Copied from groundtruth.npy")
print(f"  Channel 4: GT channel (intersection markers)")

gt_channel_idx = n_channels
gt_value_idx = 0

# Mark intersection voxels in GT channel (like sinusoid)
ground_truth_linear[gt_channel_idx, gt_value_idx, z_coords, y_coords, x_coords] = 1

print(f"  Marked {len(z_coords):,} intersection voxels as GT")

# ============================================================================
# STEP 6: Save the ground truth file
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 6: Saving groundtruth_linear.npy")
print(f"{'='*60}")

np.save('groundtruth_linear.npy', ground_truth_linear)

print(f"✓ Ground truth with 5 channels saved to: groundtruth_linear.npy")
print(f"  Final shape: {ground_truth_linear.shape}")
print(f"  Channels: 0-3 (original), 4 (GT channel)")
print(f"  GT voxels in channel 4: {np.sum(ground_truth_linear[gt_channel_idx] > 0):,}")
print(f"  Intersection = voxels where 2 or more channels overlap (method like sinusoid)")

print(f"\n{'='*60}")
print(f"Process completed successfully!")
print(f"{'='*60}")

Loaded data shape: (4, 11, 49, 344, 681)
  Channels: 4 (0-3)
  Values: 11 (0-10)
  Spatial dimensions: Z=49, Y=344, X=681

STEP 2: Finding voxel-level intersections
Channel presence shape: (4, 49, 344, 681)
  Channel 0 non-zero voxels: 1,011,360
  Channel 1 non-zero voxels: 1,968,771
  Channel 2 non-zero voxels: 772,142
  Channel 3 non-zero voxels: 655,914

Intersection statistics (voxels with 2+ channels):
  Voxels with 2 channels: 371,763
  Voxels with 3 channels: 77,518
  Voxels with 4 channels: 12,103
  Total intersection voxels: 461,384

STEP 3: Extracting intersection voxel positions
Total intersection voxels (GT): 461,384
  These are voxels where 2 or more channels overlap

STEP 4: Ground truth summary
  Total voxels in volume: 11,478,936
  Intersection voxels: 461,384
  Intersection percentage: 4.02%
  Randomly sampled 100 intersection voxels (seed=42)

STEP 5: Creating groundtruth_linear.npy with 5 channels
Created new array with shape: (5, 11, 49, 344, 681)
  Channels 0-3: Co

visulize with groundtruth

In [22]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("groundtruth_linear.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded groundtruth_linear.npy shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3: original, 4: GT spot)")

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0 - light blue
    "#1f78b4",      # channel 1 - blue
    "#b2df8a",      # channel 2 - light green
    "#33a02c",      # channel 3 - green
    "#000000",      # channel 4 - black (GT channel)
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

# Process channels 0-3 (original channels)
for ch in range(n_channels - 1):  # Process channels 0-3
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# Process channel 4 (GT spot)
gt_channel = data[n_channels - 1]  # Channel 4
# GT spots are stored at value index 0
gt_spots = gt_channel[0]  # Shape: (n_z, y_dim, x_dim)
per_channel_3d[n_channels - 1] = gt_spots.astype(np.float32)

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.5
max_points = 2000

# Plot channels 0-3 (original channels)
for ch in range(n_channels - 1):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=4,
            color=channel_colors[ch],
            opacity=0.6
        ),
        name=f"Channel {ch}"
    ))

# Plot channel 4 (GT channel) - black spots
gt_channel_idx = n_channels - 1
vol = per_channel_3d[gt_channel_idx]

if np.any(vol > 0):
    z_idx, y_idx, x_idx = np.where(vol > 0)
    
    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=4,  # Larger size for GT spots
            color=channel_colors[gt_channel_idx],  # Black
            opacity=0.8,
        ),
        name="GT spot"
    ))
    print(f"  GT spot: {len(z_idx)} spots")

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Visualization: All Channels (0-3) + GT spot",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
        bgcolor="white"
    ),
    autosize=True,
    showlegend=True,
)

print(f"\n3D Visualization Summary:")
print(f"  Channels 0-3: Original channels with colored markers")
print(f"  GT spot: Black spots (size=5)")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


Loaded groundtruth_linear.npy shape: (5, 11, 49, 344, 681)
  Channels: 5 (0-3: original, 4: GT spot)
  GT spot: 100 spots

3D Visualization Summary:
  Channels 0-3: Original channels with colored markers
  GT spot: Black spots (size=5)
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y
